# 10. Reading and Filtering Comments

Comments are where the real conversation about a ticket happens: updates from engineers, questions from the reporter, approvals from managers. Reading them from Python lets you build reports such as *"latest update on every open incident"*.

> **Before you run this notebook:** create an API token (topic 08) and set `JIRA_SERVER`, `JIRA_EMAIL` and `JIRA_API_TOKEN` as environment variables, or in Colab's **Secrets** panel. The outputs shown were recorded on a practice Jira Cloud site with a project called **myproject** (key `MYP`). Your issue keys and data will differ.

In [1]:
import os
from jira import JIRA

jira = JIRA(
    server=os.environ["JIRA_SERVER"],                 # https://your-domain.atlassian.net
    basic_auth=(os.environ["JIRA_EMAIL"], os.environ["JIRA_API_TOKEN"]),
)

In [2]:
ticket = "MYP-8"
issue = jira.issue(ticket)
comments = issue.fields.comment.comments
print(len(comments), "comments")

`issue.fields.comment.comments` is a **list** of Comment objects, oldest first.

> The list was captured when the issue was fetched. After someone adds a comment in the browser, call `jira.issue(ticket)` again to see it.

## Print Every Comment

In [3]:
for comment in comments:
    print("My comment is : ", comment.body)

My comment is :  i am working on it
My comment is :  This is my second comment
My comment is :  comment on new date


## Author and Time

In [4]:
for comment in comments:
    print("My comment is : ", comment.body)
    print("Comment Author is : ", comment.author.displayName)
    print("Comment time is : ", comment.created)
    print("---------------------------------------")

My comment is :  i am working on it
Comment Author is :  Ahmad Hassan
Comment time is :  2025-02-10T21:44:36.655+0500
---------------------------------------
My comment is :  This is my second comment
Comment Author is :  Ahmad Hassan
Comment time is :  2025-02-10T21:59:56.515+0500
---------------------------------------
My comment is :  comment on new date
Comment Author is :  Ahmad Hassan
Comment time is :  2025-02-11T01:07:48.491+0500
---------------------------------------


| Attribute | Value |
|---|---|
| `comment.body` | The comment text |
| `comment.author.displayName` | Who wrote it |
| `comment.created` | When, as an ISO 8601 string with the time zone offset |
| `comment.id` | A unique, increasing number (as a string) |

## Finding the Latest Comment

### Method 1: the Highest Comment ID

Comment IDs only increase, so the biggest ID is the newest comment.

In [5]:
for comment in comments:
    print(comment.body)
    print(comment.id)

i am working on it
10000
This is my second comment
10001
comment on new date
10002


In [6]:
commentid = 10000
for comment in comments:
    if int(comment.id) > commentid:
        commentid = int(comment.id)
print(jira.comment(issue, commentid).body)

comment on new date


`jira.comment(issue, id)` fetches a single comment by its ID.

### Method 2: the Last Item of the Loop

After a loop ends, the loop variable still holds the **last** value:

In [7]:
for comment in comments:
    latestcomments = comment.body
print(latestcomments)

comment on new date


### Method 3: Collect Them in a List

In [8]:
mycomnt = []
for comment in comments:
    latestcomments = comment.body
    mycomnt.append(latestcomments)
print(latestcomments)
print(mycomnt)

comment on new date
['i am working on it', 'This is my second comment', 'comment on new date']


In [9]:
print(mycomnt[0])      # the first (oldest) comment

i am working on it


The cleanest version uses Python indexing directly: `comments[-1].body` is the newest comment and `comments[0].body` the oldest.

## Comments by One Person

In [10]:
for comment in comments:
    eng = comment.author.displayName
    print(eng)

Ahmad Hassan
Ahmad Hassan
Ahmad Hassan


In [11]:
for comment in comments:
    eng = comment.author.displayName
    if eng == "Ahmad Hassan":
        print(comment.body)
        print(comment.created)

i am working on it
2025-02-10T21:44:36.655+0500
This is my second comment
2025-02-10T21:59:56.515+0500
comment on new date
2025-02-11T01:07:48.491+0500


> Use this on support tickets to answer questions like *"what did the developer say?"*: replace the name with the developer's display name.

## Working with the Comment Time

`comment.created` is text. Turn it into a `datetime` to show only the date or the time, or to compare dates.

### Parsing the Format

In [12]:
import datetime

timestr = "2022-11-27T11:21:53.464+0530"
datetimeobj = datetime.datetime.strptime(timestr, "%Y-%m-%dT%H:%M:%S.%f%z")
print(datetimeobj)

2022-11-27 11:21:53.464000+05:30


| Code | Part | In the example |
|---|---|---|
| `%Y-%m-%d` | Date | `2022-11-27` |
| `T` | Literal separator | `T` |
| `%H:%M:%S` | Time | `11:21:53` |
| `.%f` | Fraction of a second | `.464` |
| `%z` | Time zone offset | `+0530` |

### Only the Time or Only the Date

A shortcut: drop the last 5 characters (the `+0500` offset) and use `fromisoformat()`.

In [13]:
from datetime import datetime

for comment in comments:
    if comment.author.displayName == "Ahmad Hassan":
        print(comment.body)
        time_only = datetime.fromisoformat(comment.created[:-5]).strftime("%H:%M:%S")
        print(time_only)

i am working on it
21:44:36
This is my second comment
21:59:56
comment on new date
01:07:48


In [14]:
for comment in comments:
    if comment.author.displayName == "Ahmad Hassan":
        print(comment.body)
        date_only = datetime.fromisoformat(comment.created[:-5]).strftime("%Y-%m-%d")
        print(date_only)

i am working on it
2025-02-10
This is my second comment
2025-02-10
comment on new date
2025-02-11


### Comments from One Day

In [15]:
import datetime

for comment in comments:
    jiratime = comment.created
    datetimeobj = datetime.datetime.strptime(jiratime, "%Y-%m-%dT%H:%M:%S.%f%z")
    mycustom = datetimeobj.strftime("%Y-%m-%d")

    if mycustom == "2025-02-11":
        print(comment.body)

comment on new date


## Adding a Comment from Python

Reading is half the story. Scripts can also post updates:

In [16]:
new_comment = jira.add_comment("MYP-8", "Automated check: the login service is healthy again.")
print(new_comment.id, new_comment.body)

## Practice

1. Print the number of comments on an issue.
2. Print the author and date (no time) of the newest comment.
3. Print every comment written on or after a given date.

**Solution:**

In [17]:
from datetime import datetime

issue = jira.issue("MYP-8")
comments = issue.fields.comment.comments
print("Total comments:", len(comments))

if comments:
    newest = comments[-1]
    created = datetime.strptime(newest.created, "%Y-%m-%dT%H:%M:%S.%f%z")
    print("Newest by", newest.author.displayName, "on", created.strftime("%Y-%m-%d"))

since = "2025-02-10"
for comment in comments:
    if comment.created[:10] >= since:          # ISO dates compare correctly as text
        print(comment.created[:10], comment.body)